In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import importlib
import src.functions as fn
importlib.reload(fn)

from src.functions import keep_from_peak, length_fix  # add as you need

In [0]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
import src.functions as fn

In [0]:
# Paths
VOLUME_PATH = CFG["data"]["volume_path"]

print("Config loaded ✅")
print(f"Volume path : {VOLUME_PATH}")
print(f"Sample rate : {CFG['data']['sample_rate']} Hz")
print(f"Target length : {CFG['data']['signal_length']} samples")

In [0]:
X_train = np.load(
    f"{VOLUME_PATH}/X_raw.npy",
    allow_pickle=True
)
y_train = np.load(
    f"{VOLUME_PATH}/y_raw.npy",
    allow_pickle=True
)
X_test = np.load(
    f"{VOLUME_PATH}/X_test_raw.npy",
    allow_pickle=True
)
y_test = np.load(
    f"{VOLUME_PATH}/y_test_raw.npy",
    allow_pickle=True
)

print(f"Train signals : {X_train.shape}")
print(f"Train labels  : {y_train.shape}")
print(f"Test signals  : {X_test.shape}")
print(f"Test labels   : {y_test.shape}")

# Quick sanity check
print(f"\nFirst train signal shape : {X_train[0].shape}")
print(f"First train label        : {y_train[0]}")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Load file names from Bronze Delta table
meta_df = spark.read.table("workspace.fall_detection_project.bronze_metadata") \
               .toPandas()

# Keep only train subjects (not test)
train_file_names = meta_df["file_name"].tolist()

print(f"File names loaded : {len(train_file_names)}")
print(f"First 5:")
for name in train_file_names[:5]:
    print(f"  {name}")

In [0]:
# Load activity codes
activity_codes_train = np.load(
    f"{VOLUME_PATH}/activity_codes_train.npy",
    allow_pickle=True
)

print(f"Activity codes loaded: {activity_codes_train.shape}")

# Split X_train into activity groups
def group_by_activity(X, activity_codes, code):
    """Return list of signals matching a specific activity code."""
    return [X[i] for i in range(len(X)) if activity_codes[i] == code]

def group_by_prefix(X, activity_codes, prefix):
    """Return list of signals whose code starts with a prefix (e.g. 'F' for all falls)."""
    return [X[i] for i in range(len(X)) if activity_codes[i].startswith(prefix)]

# Falls — all F codes together
fall_data = group_by_prefix(X_train, activity_codes_train, "F")

# ADL groups
d01_data = group_by_activity(X_train, activity_codes_train, "D01")
d02_data = group_by_activity(X_train, activity_codes_train, "D02")
d03_data = group_by_activity(X_train, activity_codes_train, "D03")
d04_data = group_by_activity(X_train, activity_codes_train, "D04")
d05_data = group_by_activity(X_train, activity_codes_train, "D05")
d06_data = group_by_activity(X_train, activity_codes_train, "D06")
d07_data = group_by_activity(X_train, activity_codes_train, "D07")
d08_data = group_by_activity(X_train, activity_codes_train, "D08")
d09_data = group_by_activity(X_train, activity_codes_train, "D09")
d10_data = group_by_activity(X_train, activity_codes_train, "D10")
d11_data = group_by_activity(X_train, activity_codes_train, "D11")
d12_data = group_by_activity(X_train, activity_codes_train, "D12")
d13_data = group_by_activity(X_train, activity_codes_train, "D13")
d14_data = group_by_activity(X_train, activity_codes_train, "D14")
d15_data = group_by_activity(X_train, activity_codes_train, "D15")
d16_data = group_by_activity(X_train, activity_codes_train, "D16")
d17_data = group_by_activity(X_train, activity_codes_train, "D17")
d18_data = group_by_activity(X_train, activity_codes_train, "D18")
d19_data = group_by_activity(X_train, activity_codes_train, "D19")

# Quick summary
print(f"\nGroup sizes:")
print(f"  Falls        : {len(fall_data)}")
print(f"  D01 (walk slow)  : {len(d01_data)}")
print(f"  D02 (walk fast)  : {len(d02_data)}")
print(f"  D03 (jog slow)   : {len(d03_data)}")
print(f"  D04 (jog fast)   : {len(d04_data)}")
print(f"  D05 (stairs slow): {len(d05_data)}")
print(f"  D06 (stairs fast): {len(d06_data)}")
print(f"  D07–D19          : {sum([len(d07_data), len(d08_data), len(d09_data), len(d10_data), len(d11_data), len(d12_data), len(d13_data), len(d14_data), len(d15_data), len(d16_data), len(d17_data), len(d18_data), len(d19_data)])}")

In [0]:
print("Processing Falls...")

# Step 1 — find peak and keep 400 samples each side
fall_peaked = keep_from_peak(fall_data, window_size=400)

# Step 2 — guarantee exactly 800 samples for every signal
fall_processed = length_fix(fall_peaked, length=800)

# Step 3 — verify
shapes = set(s.shape for s in fall_processed)
print(f"\nFall signals processed : {len(fall_processed)}")
print(f"Unique shapes          : {shapes}")
print(f"Expected               : {{(6, 800)}}")
print(f"All correct            : {shapes == {(6, 800)}}")

In [0]:
from src.functions import split_and_add

# Test on one D01 signal
test_signal = d01_data[0]
print(f"Before: {test_signal.shape}")

# We need file names for split_and_add
# For now create a dummy name matching expected format
test_names = ["D01_SA01_R01.txt"]

new_data, new_names, new_codes = split_and_add(
    [test_signal],
    length=800,
    file_names=test_names
)

print(f"After split_and_add:")
print(f"  Windows created : {len(new_data)}")
print(f"  Each shape      : {new_data[0].shape}")
print(f"  Activity codes  : {set(new_codes)}")

In [0]:
import os
files = os.listdir(VOLUME_PATH)
print(files)